# HW4: PubMed RCT text classification (CPU-friendly)

Pipeline: zero-shot baseline, full finetune, linear probe, and LoRA on `armanc/pubmed-rct20k`.
Model: `epfl-llm/MedDistilBERT` with fallback to `distilbert-base-uncased`. Train subset: 30k for CPU practicality (seed 42).


In [ ]:
# If packages are missing, uncomment:
!pip install transformers datasets evaluate peft accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00


In [ ]:
!pip install -U "transformers>=4.30" "datasets>=2.14" "evaluate" "peft" "accelerate"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transforme

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
import evaluate

try:
    from peft import LoraConfig, TaskType, get_peft_model
except Exception:
    LoraConfig = None
    TaskType = None
    get_peft_model = None

set_seed(42)
np.random.seed(42)
random.seed(42)
print(f"Torch device: {'mps' if torch.backends.mps.is_available() else 'cpu'}")


Torch device: cpu


In [ ]:
cfg = {
    "dataset_name": "armanc/pubmed-rct20k",
    "model_name": "epfl-llm/MedDistilBERT",
    "fallback_model_name": "distilbert-base-uncased",
    "seed": 42,
    "max_length": 256,
    "train_subset": 30_000,
    "batch_size": 8,
    "eval_batch_size": 16,
    "num_epochs_full": 2,
    "num_epochs_probe": 2,
    "num_epochs_lora": 2,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
}


In [ ]:
raw_ds = load_dataset(cfg["dataset_name"])
print(raw_ds)

label_col = "label"
string_cols = [
    c
    for c in raw_ds["train"].column_names
    if c != label_col and getattr(raw_ds["train"].features[c], "dtype", None) == "string"
]
if not string_cols:
    raise ValueError("No text column detected")
text_col = string_cols[0]
print(f"Using text column: {text_col}")

label_feature = raw_ds["train"].features[label_col]
if hasattr(label_feature, "names") and label_feature.names:
    label_names = list(label_feature.names)
else:
    label_names = sorted(raw_ds["train"].unique(label_col))
    raw_ds = raw_ds.map(lambda x: {"label": label_names.index(x[label_col])})

label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for i, name in enumerate(label_names)}
print(f"Labels: {id2label}")

split_val = "validation" if "validation" in raw_ds else "val"
split_test = "test" if "test" in raw_ds else split_val


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/646 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/40.7M [00:00<?, ?B/s]

...

DatasetDict({
    train: Dataset({
        features: ['abstract_id', 'label', 'text', 'sentence_id'],
        num_rows: 176642
    })
    validation: Dataset({
        features: ['abstract_id', 'label', 'text', 'sentence_id'],
        num_rows: 29672
    })
    test: Dataset({
        features: ['abstract_id', 'label', 'text', 'sentence_id'],
        num_rows: 29578
    })
})
Using text column: abstract_id


Map:   0%|          | 0/176642 [00:00<?, ? examples/s]

Map:   0%|          | 0/29672 [00:00<?, ? examples/s]

Map:   0%|          | 0/29578 [00:00<?, ? examples/s]

Labels: {0: 'background', 1: 'conclusions', 2: 'methods', 3: 'objective', 4: 'results'}


In [ ]:
train_ds = raw_ds["train"].shuffle(seed=cfg["seed"]).select(range(cfg["train_subset"]))
val_ds = raw_ds[split_val]
test_ds = raw_ds[split_test]

print(train_ds)
print(val_ds)
print(test_ds)


Dataset({
    features: ['abstract_id', 'label', 'text', 'sentence_id'],
    num_rows: 30000
})
Dataset({
    features: ['abstract_id', 'label', 'text', 'sentence_id'],
    num_rows: 29672
})
Dataset({
    features: ['abstract_id', 'label', 'text', 'sentence_id'],
    num_rows: 29578
})


In [ ]:
def load_tok_and_model(primary, fallback):
    last_error = None
    for name in [primary, fallback]:
        try:
            tok = AutoTokenizer.from_pretrained(name, use_fast=True)
            model = AutoModelForSequenceClassification.from_pretrained(
                name,
                num_labels=len(label_names),
                id2label=id2label,
                label2id=label2id,
            )
            print(f"Loaded model {name}")
            return tok, model, name
        except Exception as exc:
            print(f"Failed to load {name}: {exc}")
            last_error = exc
    raise RuntimeError(f"Could not load any model. Last error: {last_error}")


tokenizer, base_model, actual_model_name = load_tok_and_model(
    cfg["model_name"], cfg["fallback_model_name"]
)


Failed to load epfl-llm/MedDistilBERT: epfl-llm/MedDistilBERT is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model distilbert-base-uncased


In [ ]:
def preprocess(batch):
    return tokenizer(batch[text_col], truncation=True, max_length=cfg["max_length"])


remove_cols = [c for c in train_ds.column_names if c != label_col]
tokenized = {
    "train": train_ds.map(preprocess, batched=True, remove_columns=remove_cols),
    split_val: val_ds.map(preprocess, batched=True, remove_columns=remove_cols),
    split_test: test_ds.map(preprocess, batched=True, remove_columns=remove_cols),
}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/29672 [00:00<?, ? examples/s]

Map:   0%|          | 0/29578 [00:00<?, ? examples/s]

In [ ]:
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M")
    return trainable, total


In [ ]:
zero_args = TrainingArguments(
    output_dir="outputs/zero_shot",
    per_device_eval_batch_size=cfg["eval_batch_size"],
    seed=cfg["seed"],
    report_to="none",
)

zero_trainer = Trainer(
    model=base_model,
    args=zero_args,
    eval_dataset=tokenized[split_val],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


zero_metrics = zero_trainer.evaluate()
print("Zero-shot metrics:", zero_metrics)
results = {"zero_shot": zero_metrics}


Zero-shot metrics: {'eval_loss': 1.612347960472107, 'eval_model_preparation_time': 0.0022, 'eval_accuracy': 0.08132245888379618, 'eval_f1_macro': 0.031690185387107314, 'eval_runtime': 25.4405, 'eval_samples_per_second': 1166.33, 'eval_steps_per_second': 72.915}


In [ ]:
def fresh_model():
    return AutoModelForSequenceClassification.from_pretrained(
        actual_model_name,
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id,
    )


def run_training(tag, model, train_dataset, eval_dataset, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    start = time.time()
    trainer.train()
    wall = time.time() - start
    eval_metrics = trainer.evaluate(eval_dataset)
    test_metrics = trainer.evaluate(tokenized[split_test], metric_key_prefix="test")
    metrics = {**eval_metrics, **test_metrics, "wall_time_sec": wall}
    results[tag] = metrics
    print(f"Run {tag} metrics:", metrics)
    return trainer, metrics


In [ ]:
full_args = TrainingArguments(
    output_dir="outputs/full_finetune",
    learning_rate=2e-5,
    per_device_train_batch_size=cfg["batch_size"],
    per_device_eval_batch_size=cfg["eval_batch_size"],
    num_train_epochs=cfg["num_epochs_full"],
    weight_decay=cfg["weight_decay"],
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    warmup_ratio=cfg["warmup_ratio"],
    seed=cfg["seed"],
    report_to="none",
)

full_model = fresh_model()
count_trainable_params(full_model)
full_trainer, full_metrics = run_training(
    "full_finetune",
    full_model,
    tokenized["train"],
    tokenized[split_val],
    full_args,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 66.96M / 66.96M


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.460878,1.453341,0.331659,0.099623
2,1.458595,1.451806,0.335501,0.100948


Run full_finetune metrics: {'eval_loss': 1.4518059492111206, 'eval_accuracy': 0.33550148287948234, 'eval_f1_macro': 0.10094847327767693, 'eval_runtime': 12.188, 'eval_samples_per_second': 2434.516, 'eval_steps_per_second': 152.198, 'epoch': 2.0, 'test_loss': 1.4568002223968506, 'test_accuracy': 0.33420109540874976, 'test_f1_macro': 0.10108746010798557, 'test_runtime': 15.7972, 'test_samples_per_second': 1872.361, 'test_steps_per_second': 117.046, 'wall_time_sec': 263.2078585624695}


In [ ]:
probe_args = TrainingArguments(
    output_dir="outputs/linear_probe",
    learning_rate=1e-3,
    per_device_train_batch_size=cfg["batch_size"],
    per_device_eval_batch_size=cfg["eval_batch_size"],
    num_train_epochs=cfg["num_epochs_probe"],
    weight_decay=0.0,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    seed=cfg["seed"],
    report_to="none",
)

probe_model = fresh_model()
head_keys = ["classifier", "score", "pre_classifier"]
for name, param in probe_model.named_parameters():
    if not any(h in name for h in head_keys):
        param.requires_grad = False
count_trainable_params(probe_model)

probe_trainer, probe_metrics = run_training(
    "linear_probe",
    probe_model,
    tokenized["train"],
    tokenized[split_val],
    probe_args,
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 0.59M / 66.96M


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.467137,1.452987,0.331659,0.099623
2,1.459091,1.452097,0.335266,0.100434


Run linear_probe metrics: {'eval_loss': 1.4520965814590454, 'eval_accuracy': 0.33526557023456455, 'eval_f1_macro': 0.10043412417970723, 'eval_runtime': 12.2705, 'eval_samples_per_second': 2418.152, 'eval_steps_per_second': 151.175, 'epoch': 2.0, 'test_loss': 1.4569282531738281, 'test_accuracy': 0.33416728649672056, 'test_f1_macro': 0.10018752217322995, 'test_runtime': 15.4885, 'test_samples_per_second': 1909.67, 'test_steps_per_second': 119.379, 'wall_time_sec': 101.72555422782898}


In [ ]:
if get_peft_model is None:
    print("PEFT not installed; run `pip install peft` to enable LoRA")
else:
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=["q_lin", "k_lin", "v_lin", "out_lin", "query", "key", "value", "dense"],
        bias="none",
    )

    lora_model = get_peft_model(fresh_model(), lora_config)
    lora_model.print_trainable_parameters()

    lora_args = TrainingArguments(
        output_dir="outputs/lora",
        learning_rate=2e-4,
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["eval_batch_size"],
        num_train_epochs=cfg["num_epochs_lora"],
        weight_decay=0.0,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        warmup_ratio=cfg["warmup_ratio"],
        seed=cfg["seed"],
        report_to="none",
    )

    lora_trainer, lora_metrics = run_training(
        "lora",
        lora_model,
        tokenized["train"],
        tokenized[split_val],
        lora_args,
    )


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 889,349 || all params: 67,846,666 || trainable%: 1.3108


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.467206,1.454276,0.331659,0.099623
2,1.454132,1.451730,0.335266,0.100434


Run lora metrics: {'eval_loss': 1.451729655265808, 'eval_accuracy': 0.33526557023456455, 'eval_f1_macro': 0.10043412417970723, 'eval_runtime': 19.7648, 'eval_samples_per_second': 1501.257, 'eval_steps_per_second': 93.854, 'epoch': 2.0, 'test_loss': 1.4568299055099487, 'test_accuracy': 0.33416728649672056, 'test_f1_macro': 0.10018752217322995, 'test_runtime': 23.8572, 'test_samples_per_second': 1239.791, 'test_steps_per_second': 77.503, 'wall_time_sec': 273.0489253997803}


In [ ]:
results


{'zero_shot': {'eval_loss': 1.612347960472107,
  'eval_model_preparation_time': 0.0022,
  'eval_accuracy': 0.08132245888379618,
  'eval_f1_macro': 0.031690185387107314,
  'eval_runtime': 25.4405,
  'eval_samples_per_second': 1166.33,
  'eval_steps_per_second': 72.915},
 'full_finetune': {'eval_loss': 1.4518059492111206,
  'eval_accuracy': 0.33550148287948234,
  'eval_f1_macro': 0.10094847327767693,
  'eval_runtime': 12.188,
  'eval_samples_per_second': 2434.516,
  'eval_steps_per_second': 152.198,
  'epoch': 2.0,
  'test_loss': 1.4568002223968506,
  'test_accuracy': 0.33420109540874976,
  'test_f1_macro': 0.10108746010798557,
  'test_runtime': 15.7972,
  'test_samples_per_second': 1872.361,
  'test_steps_per_second': 117.046,
  'wall_time_sec': 263.2078585624695},
 'linear_probe': {'eval_loss': 1.4520965814590454,
  'eval_accuracy': 0.33526557023456455,
  'eval_f1_macro': 0.10043412417970723,
  'eval_runtime': 12.2705,
  'eval_samples_per_second': 2418.152,
  'eval_steps_per_second': 1